In [1]:
from pyspark.sql import SparkSession
from delta import DeltaTable

# Initialize SparkSession with Delta Lake support
spark = SparkSession.builder \
    .appName("Hello Delta Tables") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/delta-tables") \
    .getOrCreate()

# Define the path to the Delta table within the delta-tables volume
delta_table_path = "/home/jovyan/delta-tables/hello_delta_table"

# 1. Create a Delta table
data = spark.range(0, 5)  # Create a DataFrame with a range of numbers (0-4)
data.write.format("delta").mode("overwrite").save(delta_table_path)
print("Created Delta Table")

# 2. Read from the Delta table
df = spark.read.format("delta").load(delta_table_path)
print("Reading from Delta Table:")
df.show()

# 3. Update data in the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)
delta_table.update(
    condition="id % 2 == 0",  # Update even IDs
    set={"id": "id + 100"}    # Add 100 to even IDs
)
print("Updated Delta Table:")
delta_table.toDF().show()

# 4. Delete data from the Delta table
delta_table.delete(condition="id > 100")  # Delete rows where ID is greater than 100
print("Deleted rows from Delta Table:")
delta_table.toDF().show()

# 5. Perform a Merge (Upsert)
new_data = spark.range(3, 8)  # Create new data overlapping with existing data
delta_table.alias("old").merge(
    new_data.alias("new"),
    "old.id = new.id"
).whenMatchedUpdate(set={"id": "new.id"}).whenNotMatchedInsert(values={"id": "new.id"}).execute()

print("After Merge (Upsert):")
delta_table.toDF().show()

# Stop the Spark session
spark.stop()


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-71462b05-714d-4dde-a3d2-82842712e5c6;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 386ms :: artifacts dl 18ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

Created Delta Table
Reading from Delta Table:


+---+
| id|
+---+
|  3|
|  4|
|  2|
|  1|
|  0|
+---+



Updated Delta Table:


+---+
| id|
+---+
|  3|
|104|
|  1|
|102|
|100|
+---+



Deleted rows from Delta Table:


+---+
| id|
+---+
|  1|
|100|
|  3|
+---+



After Merge (Upsert):


+---+
| id|
+---+
|  3|
|  4|
|  5|
|  6|
|  7|
|  1|
|100|
+---+



In [1]:
# This script centralizes the Spark session initialization and configuration settings.
# By defining these configurations in one place, it ensures consistency across multiple 
# notebooks or scripts that rely on Spark. This approach reduces duplication and makes 
# it easier to manage and update configurations, as changes need to be made only in this 
# file rather than in each individual notebook or script.

from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

def initialize_spark():
    builder = (SparkSession.builder
               # Set the application name for the Spark session
               .appName("read-delta-table")
               .config("spark.ui.port", "4040")
               # Specify the master URL for the Spark cluster
               .master("spark://spark-master:7077")
               # Configure the amount of memory allocated to each Spark executor
               .config("spark.executor.memory", "512m")
               # Enable Delta Lake support by adding the Delta Spark session extension
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               # Set the catalog implementation to Delta for managing metadata and table operations
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

    # Initialize and configure the Spark session with Delta Lake support
    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    
    # Set the logging level to "ERROR" to minimize log output
    spark.sparkContext.setLogLevel("ERROR")
    spark.sparkContext.cancelAllJobs()

    # Define the path within the delta-tables volume
    delta_table_path = "/home/jovyan/work/notebooks/data/delta_lake/netflix_titles"

    # Load the Spark SQL magic extension for running SQL queries with %sparksql
    get_ipython().run_line_magic('load_ext', 'sparksql_magic')
    
    # Set the output limit of Spark SQL queries to 20 rows
    get_ipython().run_line_magic('config', 'SparkSql.limit=20')
    
    # Indicate that the initialize_spark.py module is being imported
    print("notebooks/config/initialize_spark.py has been loaded!")

    return spark, delta_table_path

spark, delta_table_path = initialize_spark()


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-61528a80-21c0-4b0a-8df6-8e911a060d0c;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-core_2.12/2.4.0/delta-core_2.12-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-core_2.12;2.4.0!delta-core_2.12.jar (927ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/2.4.0/delta-storage-2.4.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;2.4.0!delta-storage.jar (51ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (110ms)
:: resolution report :: resolve 2375ms :: artifa

notebooks/config/initialize_spark.py has been loaded!


In [3]:
spark.stop()
